# TP Jour 1 — Choix des Datasets Kaggle (Mono ou Multi-Sources), HDFS & Couche Bronze

### Fiche de cadrage de votre projet

- **Titre du sujet :** Analyse des retards et des annulations de vols aux États-Unis en 2015.

- **Dataset(s) Kaggle retenu(s) (`auteur/nom-du-dataset`) :**
  1. **Source principale :** `usdot/flight-delays` — fichier `flights.csv`, contenant les informations sur les vols effectués en 2015 (dates, compagnies aériennes, aéroports, retards, annulations, distances).
  2. **Sources complémentaires pour jointure :** même dataset Kaggle, avec les fichiers `airlines.csv` (référentiel des compagnies aériennes) et `airports.csv` (référentiel des aéroports et de leurs localisations).

- **Clé(s) de jointure envisagée(s) entre sources / tables :**
  - `flights.airline` = `airlines.iata_code` : associer chaque vol à sa compagnie aérienne.
  - `flights.origin_airport` = `airports.iata_code` : identifier l'aéroport de départ.
  - `flights.destination_airport` = `airports.iata_code` : identifier l'aéroport d'arrivée.

- **Problématique métier principale :** Quels sont les facteurs associés aux retards et aux annulations des vols aux États-Unis en 2015, et comment ces perturbations varient-elles selon les compagnies aériennes, les aéroports et les périodes de l'année ?

- **Variables clés retenues (temporelles, quantitatives, catégorielles) :**
  - **Temporelles :** `year`, `month`, `day`, `day_of_week`, `scheduled_departure`, `scheduled_arrival`.
  - **Quantitatives :** `departure_delay`, `arrival_delay`, `distance`, `air_time`, `air_system_delay`, `weather_delay`.
  - **Catégorielles :** `airline`, `origin_airport`, `destination_airport`, `cancelled`, `cancellation_reason`, `diverted`.
  - **Géographiques :** `city`, `state`, `latitude`, `longitude` (issues du fichier `airports.csv`).

## 1. Téléchargement de vos Datasets Kaggle dans `/workspace/data_raw/<source>/`

In [ ]:
import re
import shutil
from pathlib import Path

DATA_RAW = Path("/workspace/data_raw")
DATA_RAW.mkdir(parents=True, exist_ok=True)
SUPPORTED_EXTS = {".csv", ".parquet", ".json", ".jsonl", ".tsv"}

# TODO : Renseignez un ou plusieurs datasets Kaggle à combiner (format "auteur/nom-du-dataset")
KAGGLE_DATASETS: list[str] = [
    "usdot/flight-delays",  # Dataset avec vols, compagnies et aéroports : jointures par codes IATA.
]

for slug in KAGGLE_DATASETS:
    slug = slug.strip()
    if not slug:
        continue
    subdir = re.sub(r"[^a-zA-Z0-9._-]+", "_", slug.split("/")[-1]).strip("_") or "dataset"
    target_dir = DATA_RAW / subdir
    existing = [f for f in target_dir.rglob("*") if f.is_file() and f.suffix.lower() in SUPPORTED_EXTS] if target_dir.exists() else []
    if not existing:
        import kagglehub
        print(f"Téléchargement de {slug} -> {target_dir}/ ...")
        cache_dir = Path(kagglehub.dataset_download(slug))
        target_dir.mkdir(parents=True, exist_ok=True)
        for src_file in sorted(cache_dir.rglob("*")):
            if src_file.is_file() and src_file.suffix.lower() in SUPPORTED_EXTS:
                dest_file = target_dir / src_file.relative_to(cache_dir)
                dest_file.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(src_file, dest_file)

local_files = sorted([f for f in DATA_RAW.rglob("*") if f.is_file() and f.suffix.lower() in SUPPORTED_EXTS])

assert len(local_files) > 0, (
    "Aucun fichier détecté dans /workspace/data_raw/. "
    "Renseignez `KAGGLE_DATASETS` ci-dessus ou utilisez `task data:download DATASET='auteur1/ds1,auteur2/ds2'`."
)

print(f"Inventaire local multi-sources dans {DATA_RAW} ({len(local_files)} fichier(s)) :")
for f in local_files:
    rel = f.relative_to(DATA_RAW)
    size_mb = f.stat().st_size / (1024 * 1024)
    expected_blocks = max(1, int(-(-f.stat().st_size // (16 * 1024 * 1024))))
    print(f" - {str(rel):<55} {size_mb:>7.2f} Mo  (~{expected_blocks} bloc(s) HDFS de 16 Mo)")

Inventaire local multi-sources dans /workspace/data_raw (3 fichier(s)) :
 - flight-delays/airlines.csv                                 0.00 Mo  (~1 bloc(s) HDFS de 16 Mo)
 - flight-delays/airports.csv                                 0.02 Mo  (~1 bloc(s) HDFS de 16 Mo)
 - flight-delays/flights.csv                                564.96 Mo  (~36 bloc(s) HDFS de 16 Mo)


## 2. Démarrage de la `SparkSession` pré-configurée sur HDFS

Grâce au fichier `config/spark-defaults.conf`, Spark pointe automatiquement vers `hdfs://namenode:9000` avec **Delta Lake** activé.

### Exercice 2.1 — Initialiser `SparkSession` et contrôler la configuration HDFS
1. Créez une session Spark nommée `"J1_HDFS_Ingestion_Bronze"`.
2. Récupérez via `spark.conf.get(...)` la valeur de `"spark.hadoop.fs.defaultFS"`, `"spark.hadoop.dfs.blocksize"` et `"spark.hadoop.dfs.replication"`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# TODO : Initialiser la SparkSession
# getOrCreate() réutilise une session existante ou en démarre une dans le conteneur Docker.
spark = SparkSession.builder.appName("J1_HDFS_Ingestion_Bronze").getOrCreate()

assert spark is not None, "TODO : Initialisez `spark` avec SparkSession.builder..."

# TODO : Lire les 3 paramètres HDFS depuis `spark.conf.get(...)`
# fs.defaultFS est l'adresse du NameNode ; blocksize est en octets ; replication = copies par bloc.
default_fs = spark.conf.get("spark.hadoop.fs.defaultFS")
block_size = spark.conf.get("spark.hadoop.dfs.blocksize")
replication = spark.conf.get("spark.hadoop.dfs.replication")

print("Spark version       :", spark.version)
print("Default FileSystem  :", default_fs)
print("HDFS Block Size (o) :", block_size)
print("HDFS Replication    :", replication)

## 3. Copie des fichiers bruts vers HDFS & Inspection des Blocs de 16 Mo

La cellule suivante téléverse vos fichiers bruts de `/workspace/data_raw` vers `/lakehouse/bronze/raw/` sur HDFS, puis interroge le NameNode pour afficher la localisation exacte de chaque bloc de 16 Mo sur les 3 DataNodes (`datanode1`, `datanode2`, `datanode3`).

In [ ]:
# On utilise l'API Java Hadoop accessible depuis PySpark pour copier les fichiers dans HDFS.
jvm = spark._jvm
hadoop_conf = spark._jsc.hadoopConfiguration()
fs = jvm.org.apache.hadoop.fs.FileSystem.get(hadoop_conf)
PathHDFS = jvm.org.apache.hadoop.fs.Path

raw_hdfs_dir = PathHDFS("/lakehouse/bronze/raw")
fs.mkdirs(raw_hdfs_dir)

for file_path in local_files:
    rel_posix = file_path.relative_to(DATA_RAW).as_posix()
    src = PathHDFS(f"file://{file_path}")
    dst = PathHDFS(f"/lakehouse/bronze/raw/{rel_posix}")
    fs.mkdirs(dst.getParent())
    fs.copyFromLocalFile(False, True, src, dst)
    print(f"Téléversé vers HDFS : {dst}")

In [ ]:
# Inspection récursive du découpage en blocs de 16 Mo sur l'ensemble des sources ingérées
# listFiles(..., True) descend aussi dans les sous-dossiers des datasets.
remote_iter = fs.listFiles(raw_hdfs_dir, True)
statuses = []
while remote_iter.hasNext():
    statuses.append(remote_iter.next())
statuses.sort(key=lambda s: s.getLen(), reverse=True)
largest_hdfs_path = statuses[0].getPath().toString()

for status in statuses:
    path = status.getPath()
    rel_hdfs = path.toString().split("/lakehouse/bronze/raw/", 1)[-1]
    length_mb = status.getLen() / (1024 * 1024)
    blocks = fs.getFileBlockLocations(status, 0, status.getLen())
    print(f"Fichier : {rel_hdfs} ({length_mb:.2f} Mo | repl={status.getReplication()})")
    for idx, blk in enumerate(blocks):
        print(f"   Bloc #{idx} ({blk.getLength() / (1024*1024):5.2f} Mo) -> DataNodes : {list(blk.getHosts())}")

### Questions d'analyse (à vérifier aussi sur http://localhost:9870)
1. Comparez sur l'interface Web du NameNode (`Utilities > Browse the file system`) le nombre de blocs de votre fichier le plus volumineux avec sa taille divisée par 16 Mo.
2. Un fichier de 3 Mo bloque-t-il réellement 16 Mo d'espace disque sur les DataNodes ? Pourquoi ?
3. Pourquoi avoir configuré `dfs.replication = 2` sur un cluster à `3` DataNodes ?

---
## 4. Crash-Test en Direct : Panne d'un DataNode et Auto-Réparation (~50s)

Dans `config/hdfs-site.xml`, le délai de détection d'un DataNode mort a été abaissé de 10m30s à **~50 secondes** :
$$2 \times \text{recheck-interval (10s)} + 10 \times \text{heartbeat.interval (3s)} = 50\text{ s}$$

### Protocole à réaliser depuis votre terminal :
1. Inspectez l'état initial des blocs : `task hdfs:fsck`
2. Arrêtez brutalement le DataNode 2 :
   ```bash
   task hdfs:crash-dn2   # équivalent à : docker stop datanode2
   ```
3. Observez l'onglet **Datanodes** sur `http://localhost:9870` pendant ~50s : dès que `datanode2` passe à **Dead**, le NameNode détecte les blocs sous-répliqués et ordonne leur copie automatique vers le 3e DataNode survivant !

### Exercice 4.1 — Vérifier la continuité de service et la *Data Locality*
Pendant que `datanode2` est éteint, lisez votre plus gros fichier (`largest_hdfs_path`) avec PySpark, affichez `df_check.count()` ainsi que `df_check.rdd.getNumPartitions()`, et expliquez le lien avec le nombre de blocs HDFS.

*(Pensez ensuite à rallumer le nœud avec `task hdfs:restore-dn2` !)*

In [ ]:
print("Fichier cible du test de résilience :", largest_hdfs_path)

# TODO : Lire `largest_hdfs_path` avec PySpark (selon son format : spark.read.csv / parquet / json)
# Le fichier flights.csv est généralement le plus gros ; on choisit la lecture selon l'extension.
# Le test doit être lancé pendant l'arrêt de datanode2, après la détection de panne.
if largest_hdfs_path.lower().endswith((".csv", ".tsv")):
    separator = "\t" if largest_hdfs_path.lower().endswith(".tsv") else ","
    df_check = spark.read.option("header", "true").option("sep", separator).csv(largest_hdfs_path)
elif largest_hdfs_path.lower().endswith(".parquet"):
    df_check = spark.read.parquet(largest_hdfs_path)
elif largest_hdfs_path.lower().endswith((".json", ".jsonl")):
    df_check = spark.read.json(largest_hdfs_path)
else:
    raise ValueError(f"Format de fichier non pris en charge : {largest_hdfs_path}")

assert df_check is not None, "TODO : Lisez `largest_hdfs_path` dans `df_check`."

print(f"Nombre total de lignes lues       : {df_check.count():,}")
print(f"Nombre de partitions Spark créées : {df_check.rdd.getNumPartitions()}")

---
## 5. Construction de votre Couche Bronze en Delta Lake (`/lakehouse/bronze/`)

### Exercice 5.1 — Convertir vos fichiers bruts (mono ou multi-datasets) en tables Delta Lake Bronze
Dans une architecture **Médaillon multi-sources**, la couche **Bronze** conserve chaque table brute fidèle à sa source tout en l'enrichissant de métadonnées de lignage (*lineage*) indispensables avant les jointures du Jour 2 :
- `_ingested_at` : `F.current_timestamp()`
- `_source_dataset` : `F.lit(dataset_name)` *(identifie le dataset d'origine)*
- `_source_file` : `F.col("_metadata.file_path")`

> **Attention aux noms de colonnes Delta/Parquet :** Si vos fichiers CSV bruts contiennent des espaces ou caractères spéciaux dans les en-têtes, normalisez-les simplement en `snake_case` (ex. `.toDF(*[c.strip().lower().replace(" ", "_") for c in df.columns])`).

Complétez la cellule ci-dessous pour ingérer chacune des tables utiles de vos datasets dans `/lakehouse/bronze/<nom_table>`.

In [ ]:
# TODO : Définissez vos tables Bronze sous la forme :
#   "nom_table_bronze": ("nom_dataset_source", "/lakehouse/bronze/raw/<sous-dossier>/<fichier>")
BRONZE_SOURCES: dict[str, tuple[str, str]] = {
    # Exemple dataset 1 (faits)    : "trips":   ("nyc_taxi", "/lakehouse/bronze/raw/nyc_taxi/trips.parquet"),
    # Exemple dataset 2 (jointure) : "weather": ("noaa_nyc", "/lakehouse/bronze/raw/noaa_nyc/weather_daily.csv"),
    # Même dataset Kaggle, trois tables reliées : vols -> compagnies et aéroports.
    "flights": ("flight-delays", "/lakehouse/bronze/raw/flight-delays/flights.csv"),
    "airlines": ("flight-delays", "/lakehouse/bronze/raw/flight-delays/airlines.csv"),
    "airports": ("flight-delays", "/lakehouse/bronze/raw/flight-delays/airports.csv"),
}

assert len(BRONZE_SOURCES) > 0, "TODO : Renseignez vos tables à ingérer dans `BRONZE_SOURCES` (idéalement >= 2 tables à joindre au Jour 2)."

for table_name, source_spec in BRONZE_SOURCES.items():
    dataset_name, raw_path = source_spec if isinstance(source_spec, tuple) else ("default", source_spec)
    bronze_delta_path = f"/lakehouse/bronze/{table_name}"

    # TODO :
    # 1. Lire `raw_path` (CSV avec header=true, inferSchema=true, ou Parquet/JSON selon la source)
    # 2. Normaliser si besoin les noms de colonnes (snake_case sans espaces)
    # 3. Ajouter `_ingested_at` (F.current_timestamp()), `_source_dataset` (F.lit(dataset_name)) et `_source_file` (F.col("_metadata.file_path"))
    # 4. Écrire au format "delta" (mode "overwrite", overwriteSchema "true") dans `bronze_delta_path`
    # 1. Lire le fichier brut ; Spark déduit les types pour les CSV.
    if raw_path.lower().endswith(".csv"):
        df_raw = spark.read.option("header", "true").option("inferSchema", "true").csv(raw_path)
    elif raw_path.lower().endswith(".tsv"):
        df_raw = spark.read.option("header", "true").option("inferSchema", "true").option("sep", "\t").csv(raw_path)
    elif raw_path.lower().endswith(".parquet"):
        df_raw = spark.read.parquet(raw_path)
    elif raw_path.lower().endswith((".json", ".jsonl")):
        df_raw = spark.read.json(raw_path)
    else:
        raise ValueError(f"Format non pris en charge : {raw_path}")

    # La colonne virtuelle _metadata.file_path indique le fichier source HDFS.
    # On la sélectionne AVANT d'écrire en Delta, sinon elle ne serait plus disponible.
    df_raw = df_raw.withColumn("_source_file", F.col("_metadata.file_path"))
    # 2. Mettre les noms de colonnes en minuscules et remplacer les caractères gênants.
    original_columns = df_raw.columns
    normalized_columns = [re.sub(r"[^a-z0-9_]+", "_", c.strip().lower()).strip("_") for c in original_columns]
    if len(set(normalized_columns)) != len(normalized_columns):
        raise ValueError(f"Noms de colonnes en conflit après normalisation : {original_columns}")
    df_raw = df_raw.toDF(*normalized_columns)
    # 3. Conserver l'heure d'ingestion et le nom du dataset pour retracer les données.
    df_raw = (df_raw
              .withColumn("_ingested_at", F.current_timestamp())
              .withColumn("_source_dataset", F.lit(dataset_name)))
    # 4. Enregistrer la table Bronze dans HDFS au format Delta Lake.
    df_raw.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(bronze_delta_path)

    assert df_raw is not None, f"TODO : Construisez et écrivez la table Bronze `{table_name}`."

    cnt = spark.read.format("delta").load(bronze_delta_path).count()
    print(f"[BRONZE OK] {bronze_delta_path:<38} (source={dataset_name:<18}) -> {cnt:>10,} lignes")

In [ ]:
# Auto-contrôle de fin de Jour 1 : au moins une table Delta valide dans /lakehouse/bronze/
bronze_root = PathHDFS("/lakehouse/bronze")
bronze_delta_tables = [
    s.getPath().getName()
    for s in fs.listStatus(bronze_root)
    if s.isDirectory() and fs.exists(PathHDFS(s.getPath(), "_delta_log"))
]
assert len(bronze_delta_tables) > 0, "Aucune table Delta détectée dans /lakehouse/bronze/ !"
print("Tables Delta Bronze validées :", bronze_delta_tables)